[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/64_reward_model_solution.ipynb)

# 🟡 Solution: Reward Model Loss

Reference solution for the reward model ranking loss used in RLHF.

Given a chosen and rejected response (as hidden states) and a scalar reward head, compute the Bradley-Terry loss:

$$\mathcal{L} = -\mathbb{E}\left[\log \sigma(r_{\text{chosen}} - r_{\text{rejected}})\right]$$

where $\sigma$ is the sigmoid function and $r = h^\top w_{\text{head}}$.

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def reward_model_loss(chosen_hidden, rejected_hidden, reward_head):
    r_chosen = (chosen_hidden @ reward_head).squeeze(-1)
    r_rejected = (rejected_hidden @ reward_head).squeeze(-1)
    margin = r_chosen - r_rejected
    loss = -torch.log(1.0 / (1.0 + torch.exp(-margin))).mean()
    return loss

In [ ]:
import math
torch.manual_seed(0)

B, D = 8, 64
reward_head = torch.randn(D, 1)

# Equal hidden states => margin = 0 => loss = -log(sigmoid(0)) = log(2)
h = torch.randn(B, D)
loss_equal = reward_model_loss(h, h, reward_head)
print(f"Equal hidden states => loss = {loss_equal.item():.4f}  (expected ≈ {math.log(2):.4f})")

# Chosen clearly better => loss should be small
chosen  = torch.ones(B, D)
rejected = -torch.ones(B, D)
loss_good = reward_model_loss(chosen, rejected, reward_head)
print(f"Chosen >> rejected  => loss = {loss_good.item():.4f}  (expected small)")

In [ ]:
from torch_judge import check
check("reward_model")